# Risk-Aware Bike Routing: Street-Level Risk Score

Data pipeline + risk model. Produces **`edge_risk.csv`**, one row per street segment in the
routing graph (`u, v, key` match `manhattan_bike.graphml`), with a `risk` column scaled 0-1.
`bike_routing.ipynb` reads that file and uses `risk` in its edge cost.

1. Download cyclist-injury crashes (NYC Open Data, Motor Vehicle Collisions - Crashes)
2. Match each crash to the street segments within 25 m of it
3. Build features per segment: crash rate, no bike lane, big road (traffic proxy), optional 311 blocked-bike-lane complaints
4. Combine them into a weighted risk score
5. **Check it**: does a score built from 2019-2023 crashes predict where 2024+ crashes happen?
6. Save `edge_risk.csv` and a risk map

Run `bike_routing.ipynb` once first so `manhattan_bike.graphml` exists.

In [1]:
import os, re
from urllib.parse import urlencode
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
import branca.colormap as cm

GRAPH_FILE = "manhattan_bike.graphml"
CRASH_FILE = "cache/bike_crashes.csv"      # raw download, cached (cache/ is gitignored)
OUT_FILE   = "edge_risk.csv"               # what the routing notebook reads

START_DATE = "2019-01-01"   # ignore crashes before this
SPLIT_DATE = "2024-01-01"   # crashes before this BUILD the score, crashes after TEST it
BUFFER_M   = 25             # a crash counts for every street segment within this many meters
MIN_LEN_M  = 50             # floor for segment length when computing rates (stops tiny segments blowing up)
CRS_M      = 32618          # UTM zone 18N, a projection in meters for NYC

#os.makedirs("cache", exist_ok=True)
G = ox.load_graphml(GRAPH_FILE)
print(f"{G.number_of_edges():,} street segments")

15,564 street segments


## 1. Download cyclist crashes

From NYC Open Data!

In [2]:
URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"
params = {
    "$select": "collision_id, crash_date, latitude, longitude, "
               "number_of_cyclist_injured, number_of_cyclist_killed",
    "$where":  f"crash_date >= '{START_DATE}T00:00:00' AND "
               "(number_of_cyclist_injured > 0 OR number_of_cyclist_killed > 0)",
    "$limit":  500000,
}

if not os.path.exists(CRASH_FILE):
    # all boroughs; borough is blank on many records, so we filter to Manhattan by location below
    pd.read_csv(URL + "?" + urlencode(params)).to_csv(CRASH_FILE, index=False)

crashes = pd.read_csv(CRASH_FILE, parse_dates=["crash_date"])
crashes = crashes.dropna(subset=["latitude", "longitude"])
crashes = crashes[(crashes.latitude != 0) & (crashes.longitude != 0)]
print(f"{len(crashes):,} cyclist-injury crashes citywide since {START_DATE}")

36,775 cyclist-injury crashes citywide since 2019-01-01


## 2. Match crashes to street segments

In [3]:
# Streets as a GeoDataFrame in meters. u->v and v->u are the same physical street,
# so crashes are counted per street and then copied to both directions.
edges = ox.graph_to_gdfs(G, nodes=False).to_crs(CRS_M)
edges["street_id"] = [(min(u, v), max(u, v), k) for u, v, k in edges.index]
streets = edges.reset_index().drop_duplicates("street_id")[["street_id", "geometry"]]

pts = gpd.GeoDataFrame(crashes, crs=4326,
                       geometry=gpd.points_from_xy(crashes.longitude, crashes.latitude)).to_crs(CRS_M)

# every (street, crash) pair within BUFFER_M. A crash at an intersection counts for each street
# that meets there, which is what we want: riding through that intersection is risky.
near = gpd.sjoin(streets, pts[["collision_id", "crash_date", "geometry"]],
                 predicate="dwithin", distance=BUFFER_M)
near["period"] = np.where(near.crash_date < SPLIT_DATE, "build", "test")
counts = near.groupby(["street_id", "period"]).size().unstack(fill_value=0)

n_manhattan = near.collision_id.nunique()
print(f"{n_manhattan:,} crashes matched to Manhattan streets "
      f"({(near.drop_duplicates('collision_id').period == 'build').sum():,} build / "
      f"{(near.drop_duplicates('collision_id').period == 'test').sum():,} test)")

11,080 crashes matched to Manhattan streets (6,999 build / 4,081 test)


## 3. Features per street segment

In [4]:
BIKE_LANE_VALUES = {"cycleway", "lane", "track", "opposite_lane", "opposite_track"}
TAG_KEYS = ["highway", "cycleway", "cycleway:left", "cycleway:right", "cycleway:both"]

def has_bike_lane(data):
    """
    Same rule as bike_routing.ipynb.
    """
    text = " ".join(str(data.get(k, "")) for k in TAG_KEYS)
    return bool(BIKE_LANE_VALUES & set(re.findall(r"[a-z_]+", text)))

def road_size(h):
    """Traffic proxy from the OSM road class until we join real traffic counts."""
    h = str(h)
    if any(t in h for t in ("trunk", "primary", "secondary")): return 1.0   # avenues, big crosstown streets
    if "tertiary" in h: return 0.5
    return 0.0                                                            # side streets, paths

years_build = (pd.Timestamp(SPLIT_DATE) - pd.Timestamp(START_DATE)).days / 365.25
years_test  = (crashes.crash_date.max() - pd.Timestamp(SPLIT_DATE)).days / 365.25

feat = edges.reset_index()[["u", "v", "key", "street_id", "length", "highway"]].copy()
feat["length"] = feat["length"].astype(float)
feat = feat.join(counts, on="street_id").fillna({"build": 0, "test": 0})
feat = feat.rename(columns={"build": "crashes_build", "test": "crashes_test"})

per_km = np.maximum(feat["length"], MIN_LEN_M) / 1000
feat["crash_rate"]      = feat.crashes_build / per_km / years_build   # crashes per km per year
feat["crash_rate_test"] = feat.crashes_test  / per_km / years_test    # only used for evaluation
lane = {(u, v, k): has_bike_lane(d) for u, v, k, d in G.edges(keys=True, data=True)}
feat["no_bike_lane"]    = [0.0 if lane[(u, v, k)] else 1.0 for u, v, k in zip(feat.u, feat.v, feat.key)]
feat["big_road"]        = feat["highway"].map(road_size)

print(f"years: build {years_build:.1f}, test {years_test:.1f}")
feat[["length", "crashes_build", "crash_rate", "no_bike_lane", "big_road"]].describe().round(2)

years: build 5.0, test 2.8


,length,crashes_build,crash_rate,no_bike_lane,big_road
count,15564.00,15564.00,15564.00,15564.00,15564.00
mean,83.55,2.41,5.72,0.77,0.37
std,97.71,3.44,8.50,0.42,0.47
min,0.92,0.00,0.00,0.00,0.00
25%,30.59,0.00,0.00,1.00,0.00
50%,75.25,1.00,2.62,1.00,0.00
75%,92.58,3.00,8.00,1.00,1.00
max,2926.21,36.00,76.01,1.00,1.00


### (Optional) 311 blocked-bike-lane complaints

Same idea as crashes. If the download fails, the risk score just leaves this feature out.

In [5]:
USE_311 = True
FILE_311 = "cache/blocked_bike_lane_311.csv"

if USE_311:
    try:
        if not os.path.exists(FILE_311):
            p = {"$select": "unique_key, created_date, latitude, longitude",
                 "$where": f"descriptor = 'Blocked Bike Lane' AND created_date >= '{START_DATE}T00:00:00' "
                           "AND borough = 'MANHATTAN'",
                 "$limit": 500000}
            pd.read_csv("https://data.cityofnewyork.us/resource/erm2-nwe9.csv?" + urlencode(p)).to_csv(FILE_311, index=False)
        c311 = pd.read_csv(FILE_311, parse_dates=["created_date"]).dropna(subset=["latitude", "longitude"])
        c311 = c311[c311.created_date < SPLIT_DATE]          # same build window as crashes
        p311 = gpd.GeoDataFrame(c311, crs=4326,
                                geometry=gpd.points_from_xy(c311.longitude, c311.latitude)).to_crs(CRS_M)
        n311 = gpd.sjoin(streets, p311[["geometry"]], predicate="dwithin", distance=BUFFER_M) \
                  .groupby("street_id").size().rename("blocked_311")
        feat = feat.drop(columns="blocked_311", errors="ignore").join(n311, on="street_id").fillna({"blocked_311": 0})
        feat["blocked_lane_rate"] = feat.blocked_311 / per_km / years_build
        print(f"{len(p311):,} blocked-bike-lane complaints joined")
    except Exception as e:
        USE_311 = False
        print("Skipping 311 complaints:", e)

29,963 blocked-bike-lane complaints joined


## 4. Risk score

Each feature is turned into a 0-1 percentile (so one extreme street can't dominate), then combined
with hand-picked weights. Risk is 0 for the safest segments and close to 1 for the riskiest.
These weights are the main thing to experiment with.

In [6]:
WEIGHTS = {
    "crash_rate":        0.55,   # past cyclist crashes nearby
    "no_bike_lane":      0.20,
    "big_road":          0.15,   # traffic proxy
    "blocked_lane_rate": 0.10,   # only used if the 311 data loaded
}

def pct(s):
    """0-1 percentile rank; zeros stay at 0."""
    return (s.rank(method="min") - 1) / max(len(s) - 1, 1) if s.max() > 0 else s * 0.0

def risk_score(feat, weights):
    used = {k: w for k, w in weights.items() if k in feat}
    total = sum(used.values())
    return sum(w * (pct(feat[k]) if k.endswith("rate") else feat[k]) for k, w in used.items()) / total

feat["risk"] = risk_score(feat, WEIGHTS)
feat["risk"].describe().round(3)

count    15564.000
mean         0.474
std          0.246
min          0.000
25%          0.200
50%          0.498
75%          0.680
max          0.998
Name: risk, dtype: float64

## 5. Does the score predict future crashes?

The score was built only from crashes **before** the split date. Here we check it against crashes
**after** it, which the score never saw. Two numbers:

- **Spearman correlation** between each street's risk and its future crash rate (higher = better ranking)
- **Top-10% capture**: take the riskiest streets that add up to 10% of total street length; what share of future crashes happened there? Random would be 10%.

Compared against simpler scores (the baselines in the proposal).

In [7]:
one_way = feat.drop_duplicates("street_id")   # count each physical street once

def capture(score, frac=0.10):
    s = one_way.assign(score=score).sort_values("score", ascending=False)
    top = s[s.length.cumsum() <= frac * s.length.sum()]
    return top.crashes_test.sum() / s.crashes_test.sum()

rng = np.random.default_rng(0)
candidates = {
    "Random":                     pd.Series(rng.random(len(one_way)), index=one_way.index),
    "No bike lane only":          one_way.no_bike_lane + 0.001 * rng.random(len(one_way)),
    "Big road only":              one_way.big_road + 0.001 * rng.random(len(one_way)),
    "Past crashes only":          one_way.crash_rate,
    "Our risk score":             one_way.risk,
}
evaluation = pd.DataFrame({
    name: {"spearman_vs_future_crashes": s.corr(one_way.crash_rate_test, method="spearman"),
           "top10pct_capture": capture(s)}
    for name, s in candidates.items()}).T.round(3)
evaluation

,spearman_vs_future_crashes,top10pct_capture
Random,0.012,0.113
No bike lane only,-0.181,0.085
Big road only,0.339,0.171
Past crashes only,0.627,0.341
Our risk score,0.589,0.242


### Weight sensitivity

How much do the results depend on the weights? (To keep the test honest, pick final weights
for a reason, e.g. this table on an earlier split, not just whatever scores best here.)

In [8]:
rows = []
for w_crash in [0.0, 0.25, 0.5, 0.75, 1.0]:
    rest = 1 - w_crash
    w = {"crash_rate": w_crash, "no_bike_lane": rest * 0.5, "big_road": rest * 0.5}
    s = risk_score(one_way, w)
    rows.append({"crash weight": w_crash, "spearman": s.corr(one_way.crash_rate_test, method="spearman"),
                 "top10pct_capture": capture(s)})
pd.DataFrame(rows).round(3)

,crash weight,spearman,top10pct_capture
0,0.00,0.140,0.147
1,0.25,0.397,0.225
2,0.50,0.527,0.228
3,0.75,0.605,0.295
4,1.00,0.627,0.341


## 6. Save for the routing notebook

In [9]:
cols = ["u", "v", "key", "risk", "crash_rate", "crashes_build", "crashes_test", "no_bike_lane", "big_road"]
if USE_311: cols.append("blocked_lane_rate")M
feat[cols].to_csv(OUT_FILE, index=False)
print(f"Saved {OUT_FILE}: {len(feat):,} rows")

Saved edge_risk.csv: 15,564 rows


## 7. Risk map

In [10]:
streets_risk = (edges.reset_index()[["street_id", "geometry"]]
                .merge(feat[["street_id", "risk", "crash_rate"]].drop_duplicates("street_id"), on="street_id")
                .drop_duplicates("street_id").to_crs(4326))
streets_risk["geometry"] = streets_risk.geometry.simplify(0.00002)
streets_risk["street_id"] = streets_risk.street_id.astype(str)

cmap = cm.LinearColormap(["#1a9850", "#fee08b", "#d73027"], vmin=0, vmax=1, caption="Street risk (0 = safest)")
m = folium.Map(location=[40.775, -73.97], zoom_start=13, tiles="cartodbpositron")
folium.GeoJson(
    streets_risk[["geometry", "risk", "crash_rate"]].round(3),
    style_function=lambda f: {"color": cmap(f["properties"]["risk"]), "weight": 2.5, "opacity": 0.9},
    tooltip=folium.GeoJsonTooltip(["risk", "crash_rate"], aliases=["Risk", "Crashes / km / yr"]),
).add_to(m)
cmap.add_to(m)
m.save("risk_map.html")
print("Saved risk_map.html")

/Users/frankbubbico/Desktop/SaveTheBikers/.venv/lib/python3.12/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


Saved risk_map.html
